In [ ]:
import datetime as dt
import sys
from pathlib import Path

# Add the workspace root to Python path so we can import from src
workspace_root = (
    Path(__file__).parent.parent.parent
    if "__file__" in globals()
    else Path.cwd().parent.parent
)
sys.path.insert(0, str(workspace_root))

import dask
import numpy as np
import pandas as pd
from typing import Union
from numpy.typing import NDArray
from numba import jit
import importlib
import random
import dask.dataframe as dd
from sqlalchemy import select, create_engine
from dotenv import load_dotenv
import os
import mc_postgres_db.models as models
from sqlalchemy.orm import Session
from coiled import Cluster
from dask import delayed
from dask.distributed import LocalCluster, Semaphore
import src.utils.stochastic as stochastic

importlib.reload(stochastic)

load_dotenv()

POSTGRES_URL = os.getenv("POSTGRES_URL")

# Define trading parameters for OU model
STOP_LOSS_FACTOR = 2.5
STOP_LOSS_PERCENTAGE = 3.5
DISCOUNT_RATE = 0.001  # Discount rate
TRANSACTION_COST = 0.001  # Transaction cost
CLUSTER_TYPE = "local"
N_WORKERS = 6
N_CONCCURENT_DATABASE_CALLS = 10

PVALUE_THRESHOLD = 0.01  # Only trade if p_value < 0.01 (99% confidence)
MU_THRESHOLD = 0.005  # Only trade if the mean-reversion rate is small enough
SIGMA_THRESHOLD = 0.01  # Only trade if the volatility is smaller enough

dask.config.set({"distributed.scheduler.locks.lease-timeout": "120s"})  # 2 minutes

engine = create_engine(POSTGRES_URL)

In [ ]:
cluster = None
if CLUSTER_TYPE == "local":
    try:
        cluster.close()
    except:
        pass
    cluster = LocalCluster(
        name="local-cluster", n_workers=N_WORKERS, memory_limit="4GB"
    )
elif CLUSTER_TYPE == "coiled":
    cluster = Cluster(
        name="coiled-cluster",
        n_workers=min(N_WORKERS * 5, 30),
        region="us-east-1",
        worker_memory="8GB",
        worker_cpu=2,
    )
    cluster.send_private_envs({"POSTGRES_URL": POSTGRES_URL})

In [ ]:
client = cluster.get_client()
display(client)

In [ ]:
def compute_exit_level(
    mu: np.ndarray,
    sigma: np.ndarray,
    theta: np.ndarray,
    discount_rate: float = 0.01,
    transaction_cost: float = 0.01,
    max_iter: int = 100,
    tol: float = 1e-7,
    max_initial_shift: int = 100,
):
    """
    Vectorized computation of the Ornstein-Uhlenbeck optimal exit level given process parameters.

    All arguments must be 1D numpy arrays of the same shape.
    Returns: 1D numpy array of exit_levels, or np.nan for positions not computed.
    """
    mu = np.asarray(mu)
    sigma = np.asarray(sigma)
    theta = np.asarray(theta)
    r = np.full_like(mu, discount_rate)
    c = np.full_like(mu, transaction_cost)

    # Function f(b)
    def f_exit_level(
        b,
        mu,
        sigma,
        theta,
        r,
        c,
    ):
        return (b - c) * stochastic.OrnsteinUhlenbeck.F_prime(
            b, mu, sigma, theta, r, use_analytical=True
        ) - stochastic.OrnsteinUhlenbeck.F(b, mu, sigma, theta, r, use_analytical=True)

    # Derivative of f(b)
    def f_prime_exit_level(
        b,
        mu,
        sigma,
        theta,
        r,
        c,
        h=1e-6,
    ):
        return (
            f_exit_level(b + h, mu, sigma, theta, r, c)
            - f_exit_level(b - h, mu, sigma, theta, r, c)
        ) / (2 * h)

    initial_guess = np.copy(theta)
    mask = f_exit_level(initial_guess, mu, sigma, theta, r, c) < 0
    shift_counter = 0
    while np.any(mask) and shift_counter < max_initial_shift:
        initial_guess[mask] += 2 * sigma[mask]
        shift_counter += 1
        mask = f_exit_level(initial_guess, mu, sigma, theta, r, c) < 0
    if np.any(mask):
        raise RuntimeError(
            f"Failed to find suitable initial guess for all points after {max_initial_shift} increments."
        )

    b = initial_guess
    converged = np.zeros_like(b, dtype=bool)
    for _ in range(max_iter):
        not_converged = ~converged
        if not np.any(not_converged):
            break
        f_val = np.zeros_like(b)
        f_prime_val = np.ones_like(b)
        f_val[not_converged] = f_exit_level(
            b[not_converged],
            mu[not_converged],
            sigma[not_converged],
            theta[not_converged],
            r[not_converged],
            c[not_converged],
        )
        f_prime_val[not_converged] = f_prime_exit_level(
            b[not_converged],
            mu[not_converged],
            sigma[not_converged],
            theta[not_converged],
            r[not_converged],
            c[not_converged],
        )
        update_mask = (f_prime_val[not_converged] != 0) & np.isfinite(
            f_prime_val[not_converged]
        )
        indices_to_update = np.flatnonzero(not_converged)[update_mask]
        b[indices_to_update] = (
            b[indices_to_update]
            - f_val[indices_to_update] / f_prime_val[indices_to_update]
        )
        converged[not_converged] = (np.abs(f_val[not_converged]) < tol) | ~update_mask
    if not np.all(converged):
        import warnings

        warnings.warn(
            f"{np.sum(~converged)} roots did not converge within {max_iter} iterations."
        )
    return b  # The computed exit levels


def compute_entry_level(
    mu: np.ndarray,
    sigma: np.ndarray,
    theta: np.ndarray,
    exit_level: np.ndarray,
    discount_rate: float,
    transaction_cost: float,
    max_iter: int = 100,
    tol: float = 1e-7,
    max_initial_shift: int = 100,
):
    """
    Compute entry_level for a partition of a Dask DataFrame. Only where pvalue is less than the threshold.
    """
    r = np.full_like(mu, discount_rate)
    c = np.full_like(mu, transaction_cost)

    def f_entry_level(x: np.ndarray, mu, sigma, theta, r, c, exit_level):
        x_neg = -x
        return stochastic.OrnsteinUhlenbeck.G(
            x_neg, mu, sigma, theta, r, use_analytical=True
        ) * (
            stochastic.OrnsteinUhlenbeck.V_prime(
                x_neg, mu, sigma, theta, r, c, exit_level, use_analytical=True
            )
            - 1
        ) - stochastic.OrnsteinUhlenbeck.G_prime(
            x_neg, mu, sigma, theta, r, use_analytical=True
        ) * (
            stochastic.OrnsteinUhlenbeck.V(
                x_neg, mu, sigma, theta, r, c, exit_level, use_analytical=True
            )
            - x_neg
            - c
        )

    def f_prime_entry_level(x: np.ndarray, mu, sigma, theta, r, c, exit_level, h=1e-6):
        return (
            f_entry_level(x + h, mu, sigma, theta, r, c, exit_level)
            - f_entry_level(x - h, mu, sigma, theta, r, c, exit_level)
        ) / (2 * h)

    initial_guess = np.copy(theta)
    mask = f_entry_level(initial_guess, mu, sigma, theta, r, c, exit_level) < 0

    shift_counter = 0
    while np.any(mask) and shift_counter < max_initial_shift:
        initial_guess[mask] += 2 * sigma[mask]
        shift_counter += 1
        mask = f_entry_level(initial_guess, mu, sigma, theta, r, c, exit_level) < 0
    if np.any(mask):
        raise RuntimeError(
            f"Failed to find suitable initial guess for all points after {max_initial_shift} increments."
        )

    d = initial_guess
    converged = np.zeros_like(d, dtype=bool)
    for _ in range(max_iter):
        mask_nc = ~converged
        if not np.any(mask_nc):
            break
        f_val = np.zeros_like(d)
        f_prime_val = np.ones_like(d)
        f_val[mask_nc] = f_entry_level(
            d[mask_nc],
            mu[mask_nc],
            sigma[mask_nc],
            theta[mask_nc],
            r[mask_nc],
            c[mask_nc],
            exit_level[mask_nc],
        )
        f_prime_val[mask_nc] = f_prime_entry_level(
            d[mask_nc],
            mu[mask_nc],
            sigma[mask_nc],
            theta[mask_nc],
            r[mask_nc],
            c[mask_nc],
            exit_level[mask_nc],
        )
        update_mask = (f_prime_val[mask_nc] != 0) & np.isfinite(f_prime_val[mask_nc])
        indices_to_update = np.flatnonzero(mask_nc)[update_mask]
        d[indices_to_update] = (
            d[indices_to_update]
            - f_val[indices_to_update] / f_prime_val[indices_to_update]
        )
        converged[mask_nc] = (np.abs(f_val[mask_nc]) < tol) | ~update_mask
    if not np.all(converged):
        import warnings

        warnings.warn(
            f"{np.sum(~converged)} roots did not converge within {max_iter} iterations."
        )

    # Entry level is -d (for short-side entry relative to theta)
    return -d

In [ ]:
@jit(nopython=True)
def compute_trades(
    timestamp: np.ndarray,
    spread: np.ndarray,
    pvalue: np.ndarray,
    mu: np.ndarray,
    sigma: np.ndarray,
    entry_level: np.ndarray,
    exit_level: np.ndarray,
    loss_level: np.ndarray,
    threshold_pvalue: float,
    threshold_mu: float,
    threshold_sigma: float,
):
    result = np.zeros_like(timestamp, dtype="int64")
    n = result.shape[0]
    trade_open = False
    for i in range(n):
        if trade_open:
            if spread[i] > exit_level[i] or spread[i] < loss_level[i]:
                result[i] = -1
                trade_open = False
        else:
            if (
                (spread[i] < entry_level[i])
                and (pvalue[i] < threshold_pvalue)
                and (spread[i] > loss_level[i])
                and (mu[i] < threshold_mu)
                and (sigma[i] < threshold_sigma)
            ):
                result[i] = 1
                trade_open = True
    return result


def calculate_pnl_with_costs(
    position_size: Union[float, int, NDArray[np.float64]],
    entry_price: Union[float, NDArray[np.float64]],
    exit_price: Union[float, NDArray[np.float64]],
    entry_time: Union[pd.Timestamp, NDArray],
    exit_time: Union[pd.Timestamp, NDArray],
    commission_rate: float = 0.001,
    borrow_rate_annual: float = 0.05,
    is_long: Union[bool, NDArray[np.bool_]] = None,
) -> Union[float, NDArray[np.float64]]:
    """
    Calculate PnL with transaction costs for long and short positions.

    Parameters
    ----------
    position_size : scalar or array
        Number of shares (can be positive, negative, or use is_long flag)
        If negative, treated as short position (unless is_long overrides)
    entry_price : scalar or array
        Entry price in $/share
    exit_price : scalar or array
        Exit price in $/share
    entry_time : pd.Timestamp or array of timestamps
        Entry timestamp for each trade
    exit_time : pd.Timestamp or array of timestamps
        Exit timestamp for each trade
    commission_rate : float, default 0.001
        Commission rate per side (e.g., 0.001 = 0.1%)
    borrow_rate_annual : float, default 0.05
        Annualized borrow cost for short positions (e.g., 0.05 = 5%)
    is_long : bool or array, optional
        Explicitly specify if position is long (True) or short (False)
        If None, inferred from sign of position_size

    Returns
    -------
    pnl : scalar or array
        Net PnL in $ (same shape as inputs)
    """
    # Convert to arrays
    position_size = np.asarray(position_size, dtype=np.float64)
    entry_price = np.asarray(entry_price, dtype=np.float64)
    exit_price = np.asarray(exit_price, dtype=np.float64)

    # Convert timestamps to numpy datetime64 if needed
    if isinstance(entry_time, pd.Timestamp):
        entry_time = np.array([entry_time], dtype="datetime64[ns]")
    elif isinstance(entry_time, (list, pd.DatetimeIndex)):
        entry_time = pd.to_datetime(entry_time).values
    else:
        entry_time = np.asarray(entry_time, dtype="datetime64[ns]")

    if isinstance(exit_time, pd.Timestamp):
        exit_time = np.array([exit_time], dtype="datetime64[ns]")
    elif isinstance(exit_time, (list, pd.DatetimeIndex)):
        exit_time = pd.to_datetime(exit_time).values
    else:
        exit_time = np.asarray(exit_time, dtype="datetime64[ns]")

    # Calculate holding period in days (including fractional days)
    time_delta = exit_time - entry_time
    holding_days = time_delta / np.timedelta64(1, "D")
    holding_days = holding_days.astype(np.float64)

    # Determine if positions are long or short
    if is_long is None:
        # Infer from sign of position_size
        is_long_arr = position_size >= 0
        abs_pos = np.abs(position_size)
    else:
        is_long_arr = np.asarray(is_long, dtype=bool)
        abs_pos = np.abs(position_size)

    # Calculate gross PnL
    pnl = np.where(
        is_long_arr,
        abs_pos * (exit_price - entry_price),  # Long PnL
        abs_pos * (entry_price - exit_price),  # Short PnL
    )

    # Commissions (both entry and exit)
    commissions = abs_pos * entry_price * commission_rate
    commissions += abs_pos * exit_price * commission_rate

    # Borrow cost (only for shorts)
    daily_rate = borrow_rate_annual / 365.0
    borrow_cost = np.where(
        ~is_long_arr,  # Only for shorts
        abs_pos * entry_price * daily_rate * holding_days,
        0.0,
    )

    net_pnl = pnl - commissions - borrow_cost

    # Return scalar if inputs were scalar
    return float(net_pnl) if net_pnl.ndim == 0 else net_pnl


def compute_pnl(
    df: pd.DataFrame,
    threshold_pvalue: float,
    threshold_mu: float,
    threshold_sigma: float,
    cash_allocation: float,  # Changed parameter name
):
    """
    Compute PnL using the beta from cointegration for proper hedging.

    Uses constant cash allocation for the LONG leg, then calculates
    the short leg position based on beta to maintain hedge.

    Model: close_1 = alpha + beta * close_2 + error
    Spread: close_1 - beta * close_2 - alpha

    When spread < 0 (entry signal):
        - close_1 is undervalued relative to close_2
        - Long $cash_allocation worth of close_1
        - Short (long_shares * beta) shares of close_2
        - This maintains the beta-based hedge from cointegration

    Parameters
    ----------
    df : pd.DataFrame
        DataFrame with trade signals and prices
    threshold_pvalue : float
        P-value threshold for entering trades
    cash_allocation : float
        Dollar amount to allocate to the LONG leg (e.g., $10,000)
        Short leg will be sized according to beta * long_shares
    """
    # Compute the enter and exit trades.
    trades = compute_trades(
        df["timestamp"].to_numpy(),
        df["spread"].to_numpy(),
        df["pvalue"].to_numpy(),
        df["mu"].to_numpy(),
        df["sigma"].to_numpy(),
        df["entry_level"].to_numpy(),
        df["exit_level"].to_numpy(),
        df["loss_level"].to_numpy(),
        threshold_pvalue,
        threshold_mu,
        threshold_sigma,
    )

    # Only take data from the frame where we are either entering or exiting a trade.
    actual_trades = df[trades != 0].copy()

    # If we have an odd number of trades, the last position is still open
    # Close it out at the last available price
    if len(actual_trades) % 2 != 0:
        # Get the last row from the original dataframe
        last_row = df.iloc[-1].copy()
        # Mark it as an exit trade by creating a single-row DataFrame
        last_row_df = pd.DataFrame([last_row])
        actual_trades = pd.concat([actual_trades, last_row_df], ignore_index=True)

    # Re-shape the arrays for shorting and longing.
    ou_mu = actual_trades["mu"].to_numpy().reshape(-1, 2)
    ou_sigma = actual_trades["sigma"].to_numpy().reshape(-1, 2)
    ou_theta = actual_trades["theta"].to_numpy().reshape(-1, 2)
    close_1_prices = actual_trades["close_1"].to_numpy().reshape(-1, 2)
    close_2_prices = actual_trades["close_2"].to_numpy().reshape(-1, 2)
    times = actual_trades["timestamp"].to_numpy().reshape(-1, 2)

    # Get the beta values at entry points (index 0 of each pair)
    betas = actual_trades["beta"].to_numpy().reshape(-1, 2)
    entry_betas = betas[:, 0]  # Use beta at entry time

    # Position sizing based on constant cash for LONG leg, beta-hedge for SHORT leg
    # Long leg: $cash_allocation / close_1_entry_price = shares
    long_position_sizes = cash_allocation / close_1_prices[:, 0]

    # Short leg: long_shares * beta = shares
    # This maintains the cointegration hedge ratio
    short_position_sizes = long_position_sizes * entry_betas

    # Compute PnL for each leg
    # Long leg: close_1
    long_pnl = calculate_pnl_with_costs(
        long_position_sizes,
        close_1_prices[:, 0],  # Entry price
        close_1_prices[:, 1],  # Exit price
        times[:, 0],
        times[:, 1],
        is_long=True,
    )

    # Short leg: close_2
    short_pnl = calculate_pnl_with_costs(
        short_position_sizes,  # beta * long_shares
        close_2_prices[:, 0],  # Entry price
        close_2_prices[:, 1],  # Exit price
        times[:, 0],
        times[:, 1],
        borrow_rate_annual=0.05,
        is_long=False,
    )

    return pd.DataFrame(
        {
            "entry_time": pd.Series(times[:, 0], dtype="datetime64[ns]"),
            "exit_time": pd.Series(times[:, 1], dtype="datetime64[ns]"),
            "ou_mu_entry": pd.Series(ou_mu[:, 0], dtype="float64"),
            "ou_mu_exit": pd.Series(ou_mu[:, 1], dtype="float64"),
            "ou_sigma_entry": pd.Series(ou_sigma[:, 0], dtype="float64"),
            "ou_sigma_exit": pd.Series(ou_sigma[:, 1], dtype="float64"),
            "ou_theta_entry": pd.Series(ou_theta[:, 0], dtype="float64"),
            "ou_theta_exit": pd.Series(ou_theta[:, 1], dtype="float64"),
            "short_entry_price": pd.Series(close_2_prices[:, 0], dtype="float64"),
            "short_exit_price": pd.Series(close_2_prices[:, 1], dtype="float64"),
            "short_pnl": pd.Series(short_pnl, dtype="float64"),
            "short_position_size": pd.Series(short_position_sizes, dtype="float64"),
            "long_entry_price": pd.Series(close_1_prices[:, 0], dtype="float64"),
            "long_exit_price": pd.Series(close_1_prices[:, 1], dtype="float64"),
            "long_pnl": pd.Series(long_pnl, dtype="float64"),
            "long_position_size": pd.Series(long_position_sizes, dtype="float64"),
            "hedge_ratio": pd.Series(entry_betas, dtype="float64"),
        }
    )

In [ ]:
@delayed
def pairs_trading_data_parition(start: dt.datetime, end: dt.datetime, provider_asset_group_id: int):
    # Compute the pricing data.
    gbm_params_1 = stochastic.GeometricBrownianMotionResult(mu=0.00001, sigma=0.001)
    gbm_1 = stochastic.GeometricBrownianMotion(gbm_params_1)
    ou_params = stochastic.OrnsteinUhlenbeckResult(mu=0.0005, sigma=0.001, theta=0.0001)
    ou = stochastic.OrnsteinUhlenbeck(ou_params)
    alpha = 0.0001
    beta = 0.5

    window_days = 7
    window = window_days * 24 * 60
    N_days = 30
    N = N_days * 24 * 30
    close_1 = gbm_1.simulate(N, N_simulations, 100)
    close_2 = alpha + beta * close_1 + ou.simulate(N, N_simulations, 0.01)

In [ ]:
gbm_params_1 = stochastic.GeometricBrownianMotionResult(mu=0.00001, sigma=0.001)
gbm_1 = stochastic.GeometricBrownianMotion(gbm_params_1)
ou_params = stochastic.OrnsteinUhlenbeckResult(mu=0.0005, sigma=0.001, theta=0.0001)
ou = stochastic.OrnsteinUhlenbeck(ou_params)
alpha = 0.0001
beta = 0.5

N_simulations = 5

window_days = 7
window = window_days * 24 * 60
N_days = 30
N = N_days * 24 * 30
close_1 = gbm_1.simulate(N, N_simulations, 100)
close_2 = alpha + beta * close_1 + ou.simulate(N, N_simulations, 0.01)

spreads = np.full((N_simulations, N), np.nan)
alphas = np.full((N_simulations, N), np.nan)
betas = np.full((N_simulations, N), np.nan)
pvalues = np.full((N_simulations, N), np.nan)
mus = np.full((N_simulations, N), np.nan)
sigmas = np.full((N_simulations, N), np.nan)
thetas = np.full((N_simulations, N), np.nan)
exit_levels = np.full((N_simulations, N), np.nan)
entry_levels = np.full((N_simulations, N), np.nan)
loss_levels = np.full((N_simulations, N), np.nan)
for i in tqdm(range(N_simulations)):
    results = stochastic.RollingCointegration(close_1[i, :], close_2[i, :], window).fit()
    alphas[i, :] = results.alpha
    betas[i, :] = results.beta
    pvalues[i, :] = results.pvalue
    ou_fit_result = stochastic.RollingOrnsteinUhlenbeck(results.alpha, results.beta, close_1[i, :], close_2[i, :], window).fit()
    mus[i, :] = ou_fit_result.mu
    sigmas[i, :] = ou_fit_result.sigma
    thetas[i, :] = ou_fit_result.theta
    exit_levels[i, :] = compute_exit_level(
        ou_fit_result.mu,
        ou_fit_result.sigma,
        ou_fit_result.theta,
        0.0001,
        0.001,
    )
    entry_levels[i, :] = compute_entry_level(
        ou_fit_result.mu,
        ou_fit_result.sigma,
        ou_fit_result.theta,
        exit_levels[i, :],
        0.0001,
        0.001,
    )
    loss_levels[i] = ou_fit_result.theta - 75 * ou_fit_result.sigma

In [ ]:
df_simulated = 

In [ ]:
plt.figure(figsize=(16, 8))

# Plot the spread line
plt.plot(
    df_simulated["timestamp"],
    df_simulated["spread"],
    label="Spread",
    linewidth=1.5,
    alpha=0.8,
    color="black",
)

# Plot entry points
plt.scatter(
    times[:, 0],
    spreads[:, 0],
    marker="^",
    color="lime",
    s=200,
    edgecolors="darkgreen",
    linewidths=2,
    zorder=5,
    label="Entry",
)

# Plot exit points
plt.scatter(
    times[:, 1],
    spreads[:, 1],
    marker="v",
    color="red",
    s=200,
    edgecolors="darkred",
    linewidths=2,
    zorder=5,
    label="Exit",
)

# Plot threshold levels
plt.axhline(ou_theta_hat, color="purple", linestyle="--", linewidth=2, label="Theta")
plt.axhline(
    ou_exit_level_hat, color="green", linestyle="--", linewidth=2, label="Exit Level"
)
plt.axhline(
    ou_entry_level_hat, color="blue", linestyle="--", linewidth=2, label="Entry Level"
)
plt.axhline(
    ou_loss_level_hat, color="orange", linestyle="--", linewidth=2, label="Loss Level"
)

plt.xlabel("Time", fontsize=12)
plt.ylabel("Spread", fontsize=12)
plt.title("Mean Reversion Trading Strategy", fontsize=14)
plt.legend(fontsize=11, loc="best")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()